Install Google Gemini, ChatGPT, Claude

In [ ]:
!pip install -q -U google-genai
!pip install openai
!pip install anthropic

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.1/53.1 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 721.9/721.9 kB 57.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 397.9/397.9 kB 33.1 MB/s eta 0:00:00


Define Library Imports

In [ ]:
import os
import pandas as pd
from google import genai
from openai import OpenAI
from anthropic import Anthropic
from google.colab import drive
from google.colab import userdata

Load dataset from GDrive

In [ ]:
# Authenticate to Google Drive to read in csv file
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Specify file path
dataset_file_path = "/content/drive/MyDrive/{folder_name}/Datasets/Dataset_5.csv"

In [ ]:
# Read data within dataset
df = pd.read_csv(dataset_file_path)

Prompt Engineering

In [ ]:
def gen_prompt(prompt_type, github_url, dev_guide_url):

  # Define gdpr articles used for privacy notice generation
  gdpr_privacy_notice_template = "https://gdpr.eu/privacy-notice/"
  gdpr_regulation = "https://gdpr-info.eu/"

  # Generate privacy notices using GDPR compliance expert persona
  if prompt_type == 1:

    # Verify system specs present
    # If both github repo link & developer guide are present for mobile app...
    if github_url != 'Not Found' and dev_guide_url != 'Not Found':
      prompt = "You are an expert in GDPR Compliance. Create a privacy notice using "+ github_url +" and or "+ dev_guide_url +" that is compliant with the requirements listed within "+ gdpr_regulation + "website and aligns with " + gdpr_privacy_notice_template
      return prompt

    # If only the developer guide is present for mobile app...
    elif github_url == 'Not Found' and dev_guide_url != 'Not Found':
      prompt = "You are an expert in GDPR Compliance. Create a privacy notice using "+ dev_guide_url +" that is compliant with the requirements listed within "+ gdpr_regulation + "website and aligns with " + gdpr_privacy_notice_template
      return prompt

    # If only the github repo link is present for mobile app...
    elif github_url != 'Not Found' and dev_guide_url == 'Not Found':
      prompt = "You are an expert in GDPR Compliance. Create a privacy notice using "+ github_url +" that is compliant with the requirements listed within "+ gdpr_regulation + "website and aligns with " + gdpr_privacy_notice_template
      return prompt

  # Generate privacy notices using mobile developer expert persona
  elif prompt_type == 2:

    # Verify system specs present
    # If both github repo link & developer guide are present for mobile app...
    if github_url != 'Not Found' and dev_guide_url != 'Not Found':
      prompt = "You are mobile app developer. Create a privacy notice using "+ github_url +" and or "+ dev_guide_url +" that is compliant with the requirements listed within "+ gdpr_regulation + "website and aligns with " + gdpr_privacy_notice_template
      return prompt

    # If only the developer guide is present for mobile app...
    elif github_url == 'Not Found' and dev_guide_url != 'Not Found':
      prompt = "You are mobile app developer. Create a privacy notice using "+ dev_guide_url +" that is compliant with the requirements listed within "+ gdpr_regulation + "website and aligns with " + gdpr_privacy_notice_template
      return prompt

    # If only the github repo link is present for mobile app...
    elif github_url != 'Not Found' and dev_guide_url == 'Not Found':
      prompt = "You are mobile app developer. Create a privacy notice using "+ github_url +" that is compliant with the requirements listed within "+ gdpr_regulation + "website and aligns with " + gdpr_privacy_notice_template
      return prompt

  # Generate privacy notices using Lawyer expert persona
  else:

    # Verify system specs present
    # If both github repo link & developer guide are present for mobile app...
    if github_url != 'Not Found' and dev_guide_url != 'Not Found':
      prompt = "You are a data privacy lawyer. Create a privacy notice using "+ github_url +" and or "+ dev_guide_url +" that is compliant with the requirements listed within "+ gdpr_regulation + "website and aligns with " + gdpr_privacy_notice_template
      return prompt

    # If only the developer guide is present for mobile app...
    elif github_url == 'Not Found' and dev_guide_url != 'Not Found':
      prompt = "You are a data privacy lawyer. Create a privacy notice using "+ dev_guide_url +" that is compliant with the requirements listed within "+ gdpr_regulation + "website and aligns with " + gdpr_privacy_notice_template
      return prompt

    # If only the github repo link is present for mobile app...
    elif github_url != 'Not Found' and dev_guide_url == 'Not Found':
      prompt = "You are a data privacy lawyer. Create a privacy notice using "+ github_url +" that is compliant with the requirements listed within "+ gdpr_regulation + "website and aligns with " + gdpr_privacy_notice_template
      return prompt

Define Func: Query Gemini

In [ ]:
def query_gemini(msg, llm_type, app_name, persona):

  # Retrieve API key
  gemini_key = userdata.get('gemini-key')

  # The client gets the API key from the environment variable
  client = genai.Client(api_key=gemini_key)

  # Query gemini
  response = client.models.generate_content(
      model=llm_type, contents=msg
  )

  # Define base directory and app folder
  base_dir = "/content/drive/MyDrive/{folder_name}/Datasets"
  app_folder = os.path.join(base_dir, app_name)

  # Create folder if it doesn't exist
  os.makedirs(app_folder, exist_ok=True)

  # Define output filename inside the app folder
  output_file = os.path.join(app_folder, f"{app_name}_{llm_type}_{persona}_privacy_notice.txt")

  # Check if file already exists
  if os.path.exists(output_file):
      print(f"File already exists, skipping: {output_file}")
      return

  # Save to file
  # 'w' mode overwrites the file. Use 'a' to append (see Option 2).
  with open(output_file, "w", encoding="utf-8") as f:
    notice = response.text

    if response.text:
      f.write(notice)
    else:
      f.write("No response returned from LLM.")

  print(f"Response saved to {output_file}")

Define Func: Query Claude

In [ ]:
def query_claude(msg, llm_type, app_name, persona):

  # Retrieve API key
  claude_key = userdata.get('claude-key')

  # The client gets the API key from the environment variable
  client = Anthropic(api_key=claude_key)

  # Query Claude
  response = client.messages.create(
    model=llm_type,
    max_tokens=1000,
    messages=[
        {"role": "user", "content": msg}
    ]
  )

  # Define base directory and app folder
  base_dir = "/content/drive/MyDrive/{folder_name}/Datasets"
  app_folder = os.path.join(base_dir, app_name)

  # Create folder if it doesn't exist
  os.makedirs(app_folder, exist_ok=True)

  # Define output filename inside the app folder
  output_file = os.path.join(app_folder, f"{app_name}_{llm_type}_{persona}_privacy_notice.txt")

  # Check if file already exists
  if os.path.exists(output_file):
      print(f"File already exists, skipping: {output_file}")
      return

  # Save to file
  # 'w' mode overwrites the file. Use 'a' to append (see Option 2).
  with open(output_file, "w", encoding="utf-8") as f:
      f.write(response.content[0].text)

  print(f"Response saved to {output_file}")

Define Func: Query ChatGPT

In [ ]:
def query_chatgpt(msg, llm_type, app_name, persona):

  # Retrieve API key
  chatgpt_key = userdata.get('chatgpt-key')

  # The client gets the API key from the environment variable
  client = OpenAI(api_key=chatgpt_key)

  # Query ChatGPT
  response = client.responses.create(
      model=llm_type,
      input=msg
  )

  # Define base directory and app folder
  base_dir = "/content/drive/MyDrive/{folder_name}/Datasets"
  app_folder = os.path.join(base_dir, app_name)

  # Create folder if it doesn't exist
  os.makedirs(app_folder, exist_ok=True)

  # Define output filename inside the app folder
  output_file = os.path.join(app_folder, f"{app_name}_{llm_type}_{persona}_privacy_notice.txt")

  # Check if file already exists
  if os.path.exists(output_file):
      print(f"File already exists, skipping: {output_file}")
      return

  # Save to file
  # 'w' mode overwrites the file. Use 'a' to append (see Option 2).
  with open(output_file, "w", encoding="utf-8") as f:
      f.write(response.output_text)

  print(f"Response saved to {output_file}")


In [ ]:
# Main Script Logic
def gemini_main(model, ptype):

  # Iterate through the dataset
  for index, row in df.iterrows():

    # Specify prompt
    msg_prompt = gen_prompt(ptype, row['Source Code'], row['Dev Guide'])

    # Gemini Privacy Notice generation
    query_gemini(msg_prompt, model, row['App Name'], ptype)

In [ ]:
# Main Script Logic
def chatgpt_main(model, ptype):

  # Iterate through the dataset
  for index, row in df.iterrows():

    # Specify prompt
    msg_prompt = gen_prompt(ptype, row['Source Code'], row['Dev Guide'])

    # ChatGPT Privacy Notice generation
    query_chatgpt(msg_prompt, model, row['App Name'], ptype)

In [ ]:
# Main Script Logic
def claude_main(model, ptype):

    # Iterate through the dataset
    for index, row in df.iterrows():

      # Specify prompt
      msg_prompt = gen_prompt(ptype, row['Source Code'], row['Dev Guide'])

      # Claude Privacy Notice generation
      query_claude(msg_prompt, model, row['App Name'], ptype)

In [ ]:
if __name__== '__main__':

  # Gemini LLM model definitions
  gemini_pro = "gemini-2.5-pro"
  gemini_flash = "gemini-2.5-flash"
  chatgpt_5 = "gpt-5"
  chatgpt_latest = "gpt-5.2"
  claude_sonnet = "claude-sonnet-4-5-20250929"
  claude_haiku = "claude-haiku-4-5-20251001"

  # Execute gemini queries
  gemini_main(gemini_flash, 1)
  gemini_main(gemini_flash, 2)
  gemini_main(gemini_flash, 3)

  gemini_main(gemini_pro, 1)
  gemini_main(gemini_pro, 2)
  gemini_main(gemini_pro, 3)

  # Execute chatgpt queries
  chatgpt_main(chatgpt_5, 1)
  chatgpt_main(chatgpt_5, 2)
  chatgpt_main(chatgpt_5, 3)

  chatgpt_main(chatgpt_latest, 1)
  chatgpt_main(chatgpt_latest, 2)
  chatgpt_main(chatgpt_latest, 3)

  # Execute claude queries
  claude_main(claude_haiku, 1)
  claude_main(claude_haiku, 2)
  claude_main(claude_haiku, 3)

  claude_main(claude_sonnet, 1)
  claude_main(claude_sonnet, 2)
  claude_main(claude_sonnet, 3)